# LAK-9 — Time travel & rollback

**Break → Detect → Fix → Prove.** A bad write — an accidental `INSERT OVERWRITE`, a wrong
`MERGE`, a fat-fingered `UPDATE` — silently corrupts the *current* table. Because every Iceberg
write is a **snapshot**, the good data isn't gone: it's still pinned by the snapshot taken just
before the bad write. Recovery is a **metadata operation** — read the old snapshot with
`VERSION AS OF` / `TIMESTAMP AS OF`, then `rollback_to_snapshot` to make it current again — no
restore-from-backup, no re-run of the pipeline.

**Pre-requisite:** the unified Spark server is up (`make up`) -> Spark UI at http://localhost:4040.
This notebook connects via Spark Connect. **Laptop-safe:** one tiny table, a handful of small
writes, all under `.tmp/`; the Teardown cell drops the table (`make clean` clears everything).

See the companion [`README.md`](./README.md) and the [troubleshooting sheet](../../docs/troubleshooting.md).
Headline tool: [`common.iceberg_meta.table_health`](../../common/iceberg_meta.py) plus the
`<t>.snapshots` metadata table. This module is the recovery counterpart to **LAK-3** (snapshot
expiration) — and the Gotcha shows how over-aggressive LAK-3 expiry can *destroy* the very
snapshot you'd roll back to.

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health, compare_health
from pyspark.sql import functions as F

T = "iceberg_catalog.default.lak9_t"   # namespace pre-created by docker-entrypoint.sh
spark

## Step 0 — Setup: good data in two snapshots (A, then B)

We create a tiny `orders` table and build up the *correct* state in two commits, then record the
snapshot ids in commit order:

- **Snapshot A** — the `CREATE` with the first 100 good orders.
- **Snapshot B** — an `append` of 50 more good orders (the "last known good" state we'll recover to).

We capture **B's `snapshot_id`** and its **`committed_at`** timestamp now — Iceberg can time-travel
by either, so later we demonstrate both `VERSION AS OF <id>` and `TIMESTAMP AS OF '<ts>'`.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")

# Snapshot A: create with 100 good orders.
good_a = (spark.range(0, 100).withColumnRenamed("id", "order_id")
          .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(10)))
          .withColumn("amount", F.round(F.rand(seed=1) * 100, 2))
          .withColumn("status", F.lit("good")))
good_a.writeTo(T).using("iceberg").create()          # snapshot A (the create)

# Snapshot B: append 50 more good orders.
good_b = (spark.range(100, 150).withColumnRenamed("id", "order_id")
          .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(10)))
          .withColumn("amount", F.round(F.rand(seed=2) * 100, 2))
          .withColumn("status", F.lit("good")))
good_b.writeTo(T).append()                           # snapshot B

# Record snapshot ids + B's timestamp (in commit order) BEFORE we break anything.
snaps  = spark.sql(f"SELECT snapshot_id, committed_at, operation FROM {T}.snapshots ORDER BY committed_at").collect()
snap_a, snap_b, ts_b = snaps[0]["snapshot_id"], snaps[1]["snapshot_id"], snaps[1]["committed_at"]

print("live rows after good load:", spark.table(T).count(), "(expected 150)")
print(f"snapshot A (create): {snap_a}")
print(f"snapshot B (append): {snap_b}   committed_at={ts_b}   <- last known good")

## 1. Break it — a destructive `INSERT OVERWRITE` with wrong data

The classic production accident: a job meant to load *today's* partition runs an unqualified
`INSERT OVERWRITE` against the whole table with the wrong (or empty) source. Atomically, in one
commit, the 150 good rows are **replaced** by 5 garbage rows. This is **snapshot C** — and it's now
the *current* table.

> Iceberg makes the overwrite atomic and instant, which is exactly why it's dangerous: there's no
> half-written state to notice, and the job "succeeds". Nothing errors. The damage is only visible
> when someone queries the table or checks the row count.

In [ ]:
# Wrong source: a tiny batch of "corrupt" rows (imagine a bad upstream / missing filter).
bad = (spark.range(0, 5).withColumnRenamed("id", "order_id")
       .withColumn("customer_id", F.lit(-1))
       .withColumn("amount", F.lit(0.0))
       .withColumn("status", F.lit("corrupt")))
bad.createOrReplaceTempView("bad")

# The accident: overwrite the ENTIRE table (no partition filter) with the wrong data.
spark.sql(f"INSERT OVERWRITE {T} SELECT * FROM bad")     # snapshot C (overwrite)

print("live rows after bad overwrite:", spark.table(T).count(), "(was 150!)")
spark.table(T).show(truncate=False)

## 2. Detect it — the snapshot log shows A / B / C, and B still has the good data

This is a **metadata** recovery, so the evidence lives in Iceberg's own metadata tables, not the
Spark UI. Two reads tell the whole story:

1. **`<t>.snapshots`** — three rows now: A (create), B (`append`), C (`overwrite`). The history is a
   complete audit trail of *what happened when*.
2. **Time-travel read of B** — `... VERSION AS OF <B>` returns **150** while the current table
   returns **5**. The good data is intact; only the *current pointer* is wrong. We read B two ways
   (by id and by timestamp) to show both work.

In [ ]:
print("snapshot history (operation per commit):")
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {T}.snapshots ORDER BY committed_at").show(truncate=False)

current_n = spark.sql(f"SELECT COUNT(*) AS n FROM {T}").first()["n"]
at_b_id   = spark.sql(f"SELECT COUNT(*) AS n FROM {T} VERSION AS OF {snap_b}").first()["n"]
ts_str    = ts_b.strftime("%Y-%m-%d %H:%M:%S.%f")
at_b_ts   = spark.sql(f"SELECT COUNT(*) AS n FROM {T} TIMESTAMP AS OF '{ts_str}'").first()["n"]

print(f"\ncurrent (snapshot C)            : {current_n} rows   <- the regression")
print(f"VERSION AS OF B (by snapshot_id): {at_b_id} rows   <- good data, recoverable")
print(f"TIMESTAMP AS OF B (by timestamp): {at_b_ts} rows   <- same, via '{ts_str}'")

print("\ngood data at B (first 3 rows):")
spark.sql(f"SELECT * FROM {T} VERSION AS OF {snap_b} ORDER BY order_id LIMIT 3").show(truncate=False)

## 3. Fix it — `rollback_to_snapshot` to B

Reading old data is diagnosis; **rollback** is the cure. `rollback_to_snapshot` resets the table's
*current* snapshot pointer back to B. Crucially, **rollback itself creates a *new* snapshot** (call
it D) whose data is identical to B's — so history is preserved and *auditable* (you can see a
rollback happened, and even roll the rollback back). It's a pure metadata commit: no data is
rewritten, so it's instant regardless of table size.

> `rollback_to_snapshot` requires the target to be an **ancestor** of the current snapshot. B is an
> ancestor of C, so this works. To jump to a *non-ancestor* snapshot (e.g. forward again, or a
> sibling branch) use `set_current_snapshot(snapshot_id => ...)` instead — same idea, fewer
> ancestry rules.

In [ ]:
res = spark.sql(f"""
    CALL iceberg_catalog.system.rollback_to_snapshot(
        table => 'default.lak9_t',
        snapshot_id => {snap_b}
    )
""")
res.show(truncate=False)   # returns (previous_snapshot_id, current_snapshot_id)

print("live rows after rollback:", spark.table(T).count(), "(expected 150 again)")
spark.sql(f"SELECT status, COUNT(*) AS n FROM {T} GROUP BY status").show()

## 4. Prove it — current state after rollback == snapshot B

Two-part proof: (1) a row-level comparison of *current-after-rollback* against *VERSION AS OF B*
shows **zero difference** — byte-for-byte the same data, and the `corrupt` rows are gone; and (2)
the rollback added a **new** snapshot (D) rather than deleting C, so the whole incident stays in the
audit log.

In [ ]:
# (1) current vs B: symmetric difference should be empty.
cur  = spark.table(T)
at_b = spark.sql(f"SELECT * FROM {T} VERSION AS OF {snap_b}")
diff = cur.exceptAll(at_b).unionAll(at_b.exceptAll(cur)).count()
corrupt_now = spark.sql(f"SELECT COUNT(*) AS n FROM {T} WHERE status = 'corrupt'").first()["n"]
print(f"row-level diff (current vs snapshot B): {diff}   (0 == identical)")
print(f"current rows: {cur.count()} | corrupt rows remaining: {corrupt_now}")

# (2) history grew (rollback added a commit); nothing was destroyed.
print("\nsnapshot history after rollback (note the new commit at the end):")
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {T}.snapshots ORDER BY committed_at").show(truncate=False)
compare_health([table_health(spark, T, "after rollback")])

## 5. Gotcha — expiry can make a snapshot **unrecoverable** (ties to LAK-3)

Rollback and time-travel only work while the target snapshot's **data files still exist**.
`expire_snapshots` (LAK-3) deletes old snapshots *and the data files only they referenced*. If
you'd already expired B — e.g. a nightly `expire_snapshots(older_than => now())` ran between the
good load and your recovery — then **B's files are gone, and rollback / `VERSION AS OF B` FAIL**.

The cell below shows the interaction safely on a **throwaway clone** (so we don't destroy the
recovered `lak9_t`): build a 2-snapshot table, expire down to the newest one, then try to
time-travel to the expired snapshot and catch the failure.

In [ ]:
G = "iceberg_catalog.default.lak9_gotcha"
spark.sql(f"DROP TABLE IF EXISTS {G}")

# Two snapshots: P (create), then Q (overwrite). P's files become reclaimable once P is expired.
(spark.range(0, 20).withColumnRenamed("id", "order_id").withColumn("status", F.lit("good"))
     .writeTo(G).using("iceberg").create())                # snapshot P
g_p = spark.sql(f"SELECT snapshot_id FROM {G}.snapshots ORDER BY committed_at").first()["snapshot_id"]
spark.sql(f"INSERT OVERWRITE {G} SELECT order_id, 'bad' FROM {G}")   # snapshot Q

# Aggressively expire: lift the 5-day age guard AND keep only the newest snapshot (drops P).
spark.sql(f"""
    CALL iceberg_catalog.system.expire_snapshots(
        table => 'default.lak9_gotcha', older_than => now(), retain_last => 1
    )
""").show(truncate=False)   # the deleted-files counts here are P's now-unrecoverable data files

# Now try to recover to P — its files are gone, so this should FAIL.
try:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {G} VERSION AS OF {g_p}").first()["n"]
    print(f"unexpectedly read {n} rows at P (expiry must not have removed it)")
except Exception as e:
    print("EXPECTED FAILURE — snapshot P is unrecoverable after expiry:")
    print(" ", str(e).splitlines()[0][:160])

spark.sql(f"DROP TABLE IF EXISTS {G}")

## 6. Takeaways & "in real production…"

- **Recovery is a metadata op.** Because every write is a snapshot, a bad overwrite/MERGE/UPDATE is
  reversible *instantly* with `rollback_to_snapshot` (ancestor) or `set_current_snapshot` (any
  snapshot) — no backup restore, no pipeline re-run, regardless of table size.
- **Diagnose with the snapshot log + time travel.** `<t>.snapshots` is your audit trail;
  `VERSION AS OF <id>` and `TIMESTAMP AS OF '<ts>'` let you *read* the good state to confirm before
  you roll back.
- **Rollback is auditable.** It creates a *new* snapshot pointing at the old data rather than
  deleting history, so the incident — and the fix — stay in the log.
- **Retention is your recovery window (the LAK-3 link).** `expire_snapshots` reclaims storage by
  deleting old snapshots *and their files*. Expire too aggressively and you delete the very snapshot
  rollback depends on — recovery then **fails**. **Size your expiry/retention to your recovery
  needs:** `history.expire.max-snapshot-age-ms` (and `min-snapshots-to-keep`) should be at least as
  long as the window in which you'd realistically need to undo a bad write. Storage cost vs.
  recoverability is a deliberate tradeoff, not a default to ignore.
- **In production:** alert on unexpected row-count drops / `overwrite` ops on critical tables; keep
  a retention window comfortably longer than your detection-to-recovery time; and rehearse rollback
  so it's muscle memory during an incident.

## 7. Teardown

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
spark.sql("DROP TABLE IF EXISTS iceberg_catalog.default.lak9_gotcha")
print("Dropped lak9_t (+ gotcha clone). (`make clean` clears all generated data under .tmp/.)")